# GARCH and GARCH-MIDAS Forecasts

Two models predict next month's log realised variance for XLI, XLB, XLE, XLY, and XLF.  
1. **GARCH(1,1):** a daily volatility model fitted separately for each ETF.
2. **GARCH-MIDAS (BAA10Y):** the same short-run GARCH component plus a macro-driven long-run component. Every parameter, including the MIDAS coefficient `theta`, is fitted separately for each ETF. BAA10Y was selected from four theory-motivated macro candidates. 

Both models use daily excess returns, a 60-month burn-in, expanding estimation windows, monthly MLE refits, and a one-month forecast horizon. The forecast origin is the current month and the target is the following month.


## Configuration

Set the input paths, ETF universe, estimation settings, and fit cache. The cache is keyed to the input data, so saved estimates are reused only when the underlying panels are unchanged.

In [1]:
import os, json, hashlib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

PROJECT_ROOT = "."
PRICE_PATH = os.path.join(PROJECT_ROOT, "daily_returns_panel_ver2.csv")
MACRO_PATH = os.path.join(PROJECT_ROOT, "final_macro_data.csv")

# Required hand-off file for the separate evaluation notebook.
OUT_TIDY = os.path.join(PROJECT_ROOT, "models", "forecasting", "forecasts", "garch_forecasts.csv")
# Each ETF's GARCH-MIDAS theta at every origin, for the theta plot in the evaluation notebook.
OUT_THETA = os.path.join(PROJECT_ROOT, "models", "forecasting", "forecasts", "garchmidas_theta_by_origin.csv")
# Cached fits make an interrupted or repeated estimation run resumable.
SUPPORT_DIR = os.path.join(PROJECT_ROOT, "supporting")
CACHE_DIR = os.path.join(SUPPORT_DIR, "saved_fits")
os.makedirs(CACHE_DIR, exist_ok=True)

TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]

# ---- shared protocol ----
SAMPLE_START = "2003-02"
BURN_IN = 60
REFIT_EVERY = 1
HORIZON = 1
K_MIDAS = 1
N_STARTS = 3
N_RANDOM = 10
DRIVER = "BAA10Y"
MIDAS_LABEL = "garch_midas_" + DRIVER.lower()
MODELS = {"garch": None, MIDAS_LABEL: DRIVER}
# GARCH-MIDAS is estimated separately for each ETF: every parameter, theta included, is ETF-specific.
MIDAS_CACHE_KEY = MIDAS_LABEL + "_sector_theta"
CACHE_ENABLED = True

for path in (PRICE_PATH, MACRO_PATH):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required input file not found: {path}")

def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# A changed input file invalidates saved parameter estimates.
DATA_FINGERPRINT = _md5(PRICE_PATH)[:12] + "_" + _md5(MACRO_PATH)[:12]
print(f"Configuration ready: {len(TICKERS)} ETFs; models = {', '.join(MODELS)}.")



Configuration ready: 5 ETFs; models = garch, garch_midas_baa10y.


## Data preparation

Load daily excess returns and the monthly BAA10Y series, align them to a common monthly sample, and calculate realised variance as the monthly sum of squared daily excess returns. January 2003 is excluded because it contains only one trading session.

In [2]:
px = pd.read_csv(PRICE_PATH, index_col="date", parse_dates=True)
need = [f"xret_{t}" for t in TICKERS]
missing = [c for c in need if c not in px.columns]
if missing:
    raise AssertionError(f"return panel is missing {missing}")
if not px.index.is_monotonic_increasing or px.index.has_duplicates:
    raise AssertionError("return panel dates are not sorted/unique")
xr_all = px[need].rename(columns={f"xret_{t}": t for t in TICKERS})
if xr_all.isna().any().any():
    raise AssertionError("excess-return panel contains NaNs")

_sess_all = pd.Series(1, index=xr_all.index).groupby(xr_all.index.to_period("M")).sum()
_jan = pd.Period("2003-01", "M")
xr = xr_all[xr_all.index >= pd.Period(SAMPLE_START, "M").start_time]

macro_all = pd.read_csv(MACRO_PATH, index_col="date", parse_dates=True)
macro_all.index = macro_all.index.to_period("M")
if not macro_all.index.is_monotonic_increasing or macro_all.index.has_duplicates:
    raise AssertionError("macro index is not sorted/unique")
if set(np.diff([p.ordinal for p in macro_all.index])) != {1}:
    raise AssertionError("macro panel has calendar gaps")
if DRIVER not in macro_all.columns:
    raise AssertionError(f"macro panel has no {DRIVER}")

# ---------------- common monthly grid ----------------
GRID = pd.period_range(max(pd.Period(SAMPLE_START, "M"), macro_all.index[0], xr.index[0].to_period("M")),
                       min(macro_all.index[-1], xr.index[-1].to_period("M")), freq="M")
N_MONTHS = len(GRID)
macro_matrix = macro_all.reindex(GRID)
if macro_matrix[DRIVER].isna().any():
    raise AssertionError(f"{DRIVER} has NaNs inside the grid")

# no partial month may remain inside the grid
_sess = pd.Series(1, index=xr.index).groupby(xr.index.to_period("M")).sum().reindex(GRID)
_wd = pd.Series({p: np.busday_count(p.start_time.date(), (p + 1).start_time.date()) for p in GRID})
_partial = list(GRID[(_sess < 0.6 * _wd).to_numpy()])
if _partial:
    raise AssertionError(f"partial months inside the grid: {_partial}")

# ---------------- target: monthly RV^2 from squared daily excess returns ----------------
rv2 = xr.pow(2).resample("ME").sum()
rv2.index = rv2.index.to_period("M")
rv2 = rv2.reindex(GRID)
if rv2.isna().any().any() or (rv2 <= 0).any().any():
    raise AssertionError("RV^2 has NaN or non-positive values inside the grid")
print(f"Data ready: {GRID[0]} to {GRID[-1]} ({N_MONTHS} months; {len(xr)} return days).")


Data ready: 2003-02 to 2025-12 (275 months; 5766 return days).


## Forecast design

Prepare the daily return arrays used by the likelihood, estimate the next month’s trading sessions from past calendar patterns, and define the expanding-window forecast calendar. Estimation begins after the 60-month burn-in and each forecast targets the following month.

In [3]:
def build_daily():
    """Concatenate the five tickers once; each fit takes index slices of these arrays."""
    pos = {p: i for i, p in enumerate(GRID)}
    rets_l, mids_l, blo, bhi = [], [], [], []
    off = 0
    for tk in TICKERS:
        s = xr[tk]
        s = s[[p in pos for p in s.index.to_period("M")]]
        rets_l.append(s.to_numpy(dtype=np.float64))
        mids_l.append(np.array([pos[p] for p in s.index.to_period("M")], dtype=np.int64))
        blo.append(off); off += len(s); bhi.append(off)
    return (np.ascontiguousarray(np.concatenate(rets_l)),
            np.ascontiguousarray(np.concatenate(mids_l)),
            np.array(blo, dtype=np.int64), np.array(bhi, dtype=np.int64))


RETS, MIDS, BLO, BHI = build_daily()
# ---- session counts, built ex ante ----
OBS = np.zeros((len(TICKERS), N_MONTHS), dtype=np.int64)
for _k in range(len(TICKERS)):
    np.add.at(OBS[_k], MIDS[BLO[_k]:BHI[_k]], 1)
WD = np.array([np.busday_count(p.start_time.date(), (p + 1).start_time.date())
               for p in GRID], dtype=np.int64)
MOY = np.array([p.month for p in GRID], dtype=np.int64)


def estimate_sessions(target_idx, obs_k, min_share=0.6):
    """Sessions in month `target_idx`, using ONLY months strictly before it: weekday count
    (pure calendar) minus the average holiday shortfall in earlier months of the same
    calendar month. The target month's own returns are never touched."""
    if target_idx <= 0:
        return int(WD[target_idx])
    o, w, m = obs_k[:target_idx], WD[:target_idx], MOY[:target_idx]
    keep = o >= min_share * w
    if not np.any(keep):
        return int(WD[target_idx])
    short = (w - o)[keep]
    same = short[m[keep] == MOY[target_idx]]
    sf = float(np.mean(same)) if len(same) else float(np.mean(short))
    return int(min(max(int(round(WD[target_idx] - sf)), 1), int(WD[target_idx])))


def slice_bounds(K, n_months_obs):
    """Daily slice [lo, hi) per ticker covering months K .. n_months_obs-1. Starting at
    month K guarantees every month the filter sees has a complete MIDAS lookback."""
    lo = np.empty(len(TICKERS), dtype=np.int64)
    hi = np.empty(len(TICKERS), dtype=np.int64)
    for k in range(len(TICKERS)):
        seg = MIDS[BLO[k]:BHI[k]]
        lo[k] = BLO[k] + int(np.searchsorted(seg, K, side="left"))
        hi[k] = BLO[k] + int(np.searchsorted(seg, n_months_obs, side="left"))
    return lo, hi


# ---- the protocol dates, derived rather than typed ----
FIRST_ORIGIN = K_MIDAS + BURN_IN - 1          # estimation months K .. FIRST_ORIGIN = BURN_IN
LAST_ORIGIN = N_MONTHS - 2                    # its target is the last grid month
ORIGINS = list(range(FIRST_ORIGIN, LAST_ORIGIN + 1))
TARGETS = pd.PeriodIndex([GRID[o + HORIZON] for o in ORIGINS])
print(f"Forecast design ready: {len(TARGETS)} monthly forecasts per ETF, from {TARGETS[0]} to {TARGETS[-1]}.")


Forecast design ready: 214 monthly forecasts per ETF, from 2008-03 to 2025-12.


## Model definition

Define the shared daily GARCH recursion and the GARCH-MIDAS extension. GARCH sets the macro effect to zero; GARCH-MIDAS estimates the BAA10Y long-run component separately for each ETF.

In [4]:
try:
    from numba import njit
except Exception:
    def njit(*a, **k):
        def deco(f):
            return f
        return deco if not a else a[0]

LOG2PI = float(np.log(2.0 * np.pi))
MAX_PERSIST = 0.9995
LOG_TAU_MIN, LOG_TAU_MAX = -30.0, 5.0
Z_CLAMP = 25.0
OMEGA2_MIN, OMEGA2_MAX = 1.001, 50.0
OMEGA2_FIXED_K1 = 2.0          # K=1 leaves omega2 unidentified; pin it and drop it from the count


@njit(cache=True)
def _sig(x):
    if x > Z_CLAMP:
        x = Z_CLAMP
    elif x < -Z_CLAMP:
        x = -Z_CLAMP
    return 1.0 / (1.0 + np.exp(-x))


def _logit(p):
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    return float(np.log(p / (1 - p)))


@njit(cache=True)
def midas_weights(K, omega2):
    """Beta weights w_j propto (1 - j/(K+1))^(omega2-1), j = 1..K, normalised to sum 1.

    At K = 1 this is [1.0] for every omega2: the shape parameter has no effect.
    """
    w = np.empty(K)
    for j in range(1, K + 1):
        base = 1.0 - j / (K + 1.0)
        if base <= 0.0:
            base = 1e-12
        w[j - 1] = base ** (omega2 - 1.0)
    s = w.sum()
    if s <= 0.0 or not np.isfinite(s):
        for j in range(K):
            w[j] = 1.0 / K
        return w
    return w / s


@njit(cache=True)
def midas_index(x, K, omega2, n_months):
    """mid[t] = sum_{j=1..K} w_j * x[t-j]. Defined (and used) only for t >= K."""
    w = midas_weights(K, omega2)
    out = np.zeros(n_months)
    for t in range(K, n_months):
        s = 0.0
        for j in range(1, K + 1):
            s += w[j - 1] * x[t - j]
        out[t] = s
    return out


@njit(cache=True)
def filter_core(rets, mids, tau, mu, alpha, beta, lo, hi):
    """GARCH recursion over daily slice [lo, hi). Returns
    (loglik, n_used, g_last, resid_last_sq, ok).

    g_last is the g of the final observed day; resid_last_sq is (r_last - mu)^2. Together
    they are exactly what the one-step forecast needs.
    """
    n = hi - lo
    if n < 2:
        return 0.0, 0, 1.0, 0.0, False
    g = 1.0
    ll = 0.0
    used = 0
    resid_prev_sq = 0.0
    for i in range(lo, hi):
        tau_t = tau[mids[i]]
        if tau_t != tau_t or tau_t <= 0.0:
            return 0.0, 0, 1.0, 0.0, False
        if i > lo:
            g = (1.0 - alpha - beta) + alpha * resid_prev_sq / tau_t + beta * g
            if g < 1e-10:
                g = 1e-10
            elif g > 1e10:
                g = 1e10
        r = rets[i] - mu
        resid_prev_sq = r * r
        v = g * tau_t
        if v < 1e-16:
            v = 1e-16
        ll += -0.5 * (LOG2PI + np.log(v) + r * r / v)
        used += 1
    return ll, used, g, resid_prev_sq, True


@njit(cache=True)
def nll_pooled(z, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta):
    theta = z[4 * n_tick] if use_theta == 1 else 0.0
    if use_theta == 1:
        omega2 = OMEGA2_MIN + (OMEGA2_MAX - OMEGA2_MIN) * _sig(z[4 * n_tick + 1])
        mid = midas_index(xs, K, omega2, n_months)
    else:
        mid = np.zeros(n_months)

    total = 0.0
    for k in range(n_tick):
        mu = z[k]
        persist = MAX_PERSIST * _sig(z[n_tick + k])
        split = _sig(z[2 * n_tick + k])
        alpha = persist * split
        beta = persist * (1.0 - split)
        m = z[3 * n_tick + k]
        tau = np.empty(n_months)
        for t in range(n_months):
            lt = m + theta * mid[t]
            if lt < LOG_TAU_MIN:
                lt = LOG_TAU_MIN
            elif lt > LOG_TAU_MAX:
                lt = LOG_TAU_MAX
            tau[t] = np.exp(lt)
        ll, used, _, _, ok = filter_core(rets, mids, tau, mu, alpha, beta, starts[k], ends[k])
        if (not ok) or used == 0 or ll != ll:
            return 1e12
        total += ll
    if total != total:
        return 1e12
    return -total


@njit(cache=True)
def expected_g_sum(g1, rho, n_days):
    """E[sum_{k=1..n} g_k] given a deterministic g_1, since E[g_k] = 1 + rho^{k-1}(g_1 - 1)."""
    if n_days <= 0:
        return 0.0
    if abs(1.0 - rho) < 1e-12:
        return n_days * g1
    return n_days + (g1 - 1.0) * (1.0 - rho ** n_days) / (1.0 - rho)


@njit(cache=True)
def nll_grad(z, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta):
    """Central-difference gradient of nll_pooled. Accurate to O(h^2), far more precise than
    the one-sided default, which is what lets the optimiser reach the exact maximum."""
    gr = np.zeros(z.shape[0])
    for i in range(z.shape[0]):
        h = 1e-5 * max(1.0, abs(z[i]))
        zp = z.copy()
        zp[i] += h
        zm = z.copy()
        zm[i] -= h
        gr[i] = (nll_pooled(zp, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta)
                 - nll_pooled(zm, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta)) / (2.0 * h)
    return gr

print("Model core ready.")


Model core ready.


In [5]:
def polish(z0, args):
    """Climb from z0 to the exact top of the likelihood.

    Each parameter is first rescaled by its own curvature (diagonal preconditioning), because
    mu is about a thousand times more sensitive than the other parameters; without this the
    optimiser crawls. It then runs L-BFGS-B with the accurate gradient and tight tolerances.
    Returns (z, negative log-likelihood).
    """
    f0 = nll_pooled(z0, *args)
    s = np.ones_like(z0)
    for i in range(len(z0)):
        h = 1e-4 * max(1.0, abs(z0[i]))
        zp = z0.copy(); zp[i] += h
        zm = z0.copy(); zm[i] -= h
        curv = (nll_pooled(zp, *args) - 2.0 * f0 + nll_pooled(zm, *args)) / h ** 2
        if curv > 1e-8:
            s[i] = 1.0 / np.sqrt(curv)
    r = minimize(lambda u: nll_pooled(z0 + s * u, *args), np.zeros_like(z0),
                 jac=lambda u: s * nll_grad(z0 + s * u, *args), method="L-BFGS-B",
                 options=dict(maxiter=20000, maxfun=400000, ftol=1e-15, gtol=1e-9))
    if np.isfinite(r.fun) and r.fun <= f0:
        return z0 + s * r.x, float(r.fun)
    return z0, float(f0)


class PooledGarchMidas:
    """GARCH-MIDAS with theta and omega2 POOLED across tickers; mu, alpha, beta, m per ticker.

    use_theta=False pins theta to 0, making tau a per-ticker constant and recovering
    GARCH(1,1) exactly. Both models share this class, likelihood, optimiser and forecast,
    so they cannot differ through an implementation detail rather than an economic one.
    """

    def __init__(self, K, tickers, use_theta=True, n_starts=3, seed=0, n_random=0, cap_macro=False):
        self.K = int(K)
        self.tickers = list(tickers)
        self.n_tick = len(self.tickers)
        self.use_theta = bool(use_theta)
        self.n_starts = int(n_starts)
        self.seed = int(seed)
        self.n_random = int(n_random)
        # Diagnostic option, off in the main results: when forecasting, hold the macro index
        # inside the range it took during the estimation window (no extrapolation).
        self.cap_macro = bool(cap_macro)
        self.params = None
        self.x_mean, self.x_std = 0.0, 1.0
        self.fit_ok, self.loglik = False, np.nan
        self.conv_gap = np.nan

    @property
    def n_free_params(self):
        if not self.use_theta:
            return 4 * self.n_tick
        return 4 * self.n_tick + (2 if self.K > 1 else 1)

    @property
    def aic(self):
        return np.nan if not np.isfinite(self.loglik) else 2 * self.n_free_params - 2 * self.loglik

    def _standardise(self, x_raw, freeze=False):
        x = np.asarray(x_raw, dtype=np.float64)
        if freeze:
            sd = float(np.std(x))
            self.x_mean = float(np.mean(x))
            self.x_std = sd if (np.isfinite(sd) and sd > 1e-12) else 1.0
        return (x - self.x_mean) / self.x_std

    def fit(self, rets, mids, starts, ends, macro_raw, n_months, extra_z0=None):
        """macro_raw must hold months 0..n_months-1 and no more (leak guard).

        extra_z0: optional warm-start vectors (e.g. last month's solution, or the nested
        GARCH solution with theta = 0). They are only starting points for the optimiser;
        they carry no data, so they cannot leak information."""
        if len(macro_raw) < n_months:
            raise ValueError(f"macro_raw has {len(macro_raw)} months, need {n_months}")
        for k in range(self.n_tick):
            seg = mids[starts[k]:ends[k]]
            if len(seg) and int(seg.max()) >= n_months:
                raise ValueError("daily data reaches the target month (leak guard)")

        self._standardise(macro_raw[:n_months], freeze=True)
        xs = self._standardise(macro_raw[:n_months])
        ut = 1 if self.use_theta else 0
        args = (rets, mids, starts, ends, xs, self.K, int(n_months), self.n_tick, ut)

        base_mu = np.array([float(np.mean(rets[starts[k]:ends[k]])) for k in range(self.n_tick)])
        base_m = np.array([float(np.log(max(np.var(rets[starts[k]:ends[k]]), 1e-12)))
                           for k in range(self.n_tick)])
        grid = [(0.06, 0.92, 0.0, 3.0), (0.10, 0.85, -0.25, 6.0), (0.03, 0.95, 0.25, 1.5)]
        z0s = []
        for (a, b, th, w2) in grid[:max(1, self.n_starts)]:
            z0s.append(np.concatenate([
                base_mu,
                np.full(self.n_tick, _logit((a + b) / MAX_PERSIST)),
                np.full(self.n_tick, _logit(a / (a + b))),
                base_m,
                [th, _logit((w2 - OMEGA2_MIN) / (OMEGA2_MAX - OMEGA2_MIN))]]))
        if extra_z0:
            z0s.extend(np.asarray(z, dtype=np.float64) for z in extra_z0)
        # Random starting points, spread over the plausible range. The likelihood can have
        # several peaks, and starting in many different places makes it very likely that one
        # of the searches reaches the highest. The seed is fixed, so every run is reproducible.
        # Starting points contain no data, so they cannot leak information.
        rng_s = np.random.default_rng(self.seed)
        for _ in range(self.n_random):
            a = rng_s.uniform(0.02, 0.15, self.n_tick)
            b = rng_s.uniform(0.80, 0.985 - a)
            z0s.append(np.concatenate([
                base_mu,
                [_logit(v) for v in (a + b) / MAX_PERSIST],
                [_logit(v) for v in a / (a + b)],
                base_m + rng_s.uniform(-1.0, 1.0, self.n_tick),
                [float(rng_s.uniform(-0.6, 0.6)) if self.use_theta else 0.0,
                 _logit((OMEGA2_FIXED_K1 - OMEGA2_MIN) / (OMEGA2_MAX - OMEGA2_MIN))]]))

        res = [minimize(nll_pooled, z0, args=args, method="L-BFGS-B",
                        options=dict(maxiter=800)) for z0 in z0s]
        good = [r for r in res if r.success and np.isfinite(r.fun) and r.fun < 1e11]
        if not good:
            rng = np.random.default_rng(self.seed)
            for _ in range(4):
                a = float(rng.uniform(0.03, 0.14))
                b = float(rng.uniform(0.75, 0.95 - a))
                z0 = np.concatenate([
                    base_mu, np.full(self.n_tick, _logit((a + b) / MAX_PERSIST)),
                    np.full(self.n_tick, _logit(a / (a + b))), base_m,
                    [float(rng.uniform(-0.4, 0.4)),
                     _logit((float(rng.uniform(1.3, 12.0)) - OMEGA2_MIN)
                            / (OMEGA2_MAX - OMEGA2_MIN))]])
                r = minimize(nll_pooled, z0, args=args, method="L-BFGS-B",
                             options=dict(maxiter=800))
                res.append(r)
                if r.success and np.isfinite(r.fun) and r.fun < 1e11:
                    good.append(r)
                    break

        # Maximum likelihood = the lowest negative log-likelihood among ALL finite runs.
        # L-BFGS-B can report success=False after a line-search stop at a point that is
        # still the best one found; discarding it would throw away the actual maximum.
        pool = [r for r in res if np.isfinite(r.fun) and r.fun < 1e11]
        if not pool:
            self.fit_ok, self.params, self.loglik = False, None, np.nan
            return self
        # Climb EVERY candidate to its exact top, then keep the highest. The likelihood can
        # have more than one peak, so stopping any candidate early could pick the wrong one.
        tops = [polish(r.x, args) for r in pool]
        z, f_best = min(tops, key=lambda t: t[1])
        # Convergence certificate: a second climb from the winner must find nothing more.
        z2, f2 = polish(z, args)
        self.conv_gap = float(max(f_best - f2, 0.0))
        if f2 < f_best:
            z, f_best = z2, f2
        self.fit_ok = bool(np.isfinite(f_best))
        n = self.n_tick
        per = {}
        for k, tk in enumerate(self.tickers):
            persist = MAX_PERSIST * _sig(z[n + k])
            split = _sig(z[2 * n + k])
            per[tk] = dict(mu=float(z[k]), alpha=float(persist * split),
                           beta=float(persist * (1 - split)), m=float(z[3 * n + k]))
        om = (float(OMEGA2_MIN + (OMEGA2_MAX - OMEGA2_MIN) * _sig(z[4 * n + 1]))
              if (self.use_theta and self.K > 1) else float(OMEGA2_FIXED_K1))
        self.params = dict(per_ticker=per,
                           theta=float(z[4 * n]) if self.use_theta else 0.0,
                           omega2=om, x_mean=self.x_mean, x_std=self.x_std,
                           K=self.K, use_theta=self.use_theta)
        self.loglik = float(-f_best)
        return self

    def set_params(self, p):
        self.params = dict(p)
        self.params["per_ticker"] = {k: dict(v) for k, v in p["per_ticker"].items()}
        self.x_mean, self.x_std = p["x_mean"], p["x_std"]
        self.K = int(p["K"])
        self.use_theta = bool(p["use_theta"])
        self.fit_ok = True
        return self

    def params_to_z(self, p, theta=None):
        """Map a parameter dict back to the optimiser's unconstrained vector (warm start)."""
        n = self.n_tick
        z = np.empty(4 * n + 2)
        for k, tk in enumerate(self.tickers):
            t = p["per_ticker"][tk]
            persist = t["alpha"] + t["beta"]
            z[k] = t["mu"]
            z[n + k] = _logit(persist / MAX_PERSIST)
            z[2 * n + k] = _logit(t["alpha"] / persist)
            z[3 * n + k] = t["m"]
        z[4 * n] = p["theta"] if theta is None else theta
        z[4 * n + 1] = _logit((p.get("omega2", OMEGA2_FIXED_K1) - OMEGA2_MIN) / (OMEGA2_MAX - OMEGA2_MIN))
        return z

    def forecast_h1(self, ticker, rets, mids, lo, hi, macro_raw, origin_idx, n_days_target):
        """E[sum of squared daily returns] in month origin_idx+1, using only data through
        month origin_idx.

        At h = 1 the target month's MIDAS sum reaches months origin_idx down to
        origin_idx+1-K, every one of which is observable at the origin, so no truncation
        or weight renormalisation is required.
        """
        p = self.params
        if p is None:
            return np.nan
        tp = p["per_ticker"][ticker]
        target_idx = origin_idx + 1
        if hi > lo and int(mids[lo:hi].max()) > origin_idx:
            raise ValueError("training days reach past the origin month (leak guard)")
        if len(macro_raw) < origin_idx + 1:
            raise ValueError("macro array does not cover the origin month")
        if target_idx < self.K or n_days_target <= 0:
            return np.nan

        n_obs = origin_idx + 1
        xs = self._standardise(macro_raw[:n_obs])
        if self.use_theta:
            w = midas_weights(self.K, p["omega2"])
            mid_t = 0.0
            for j in range(1, self.K + 1):
                idx = target_idx - j
                if idx < 0 or idx > origin_idx:
                    raise ValueError("MIDAS lag reaches an unobservable month (leak guard)")
                mid_t += w[j - 1] * xs[idx]
            mid_in = midas_index(xs, self.K, p["omega2"], n_obs)
            if self.cap_macro:
                mid_t = min(max(mid_t, float(mid_in[self.K:n_obs].min())), float(mid_in[self.K:n_obs].max()))
        else:
            mid_t = 0.0
            mid_in = np.zeros(n_obs)

        tau_target = float(np.exp(np.clip(tp["m"] + p["theta"] * mid_t,
                                          LOG_TAU_MIN, LOG_TAU_MAX)))
        if not np.isfinite(tau_target) or tau_target <= 0:
            return np.nan

        tau_in = np.exp(np.clip(tp["m"] + p["theta"] * mid_in, LOG_TAU_MIN, LOG_TAU_MAX))
        _, _, g_last, resid_last_sq, ok = filter_core(
            rets, mids, tau_in, tp["mu"], tp["alpha"], tp["beta"], lo, hi)
        if not ok:
            return np.nan

        rho = tp["alpha"] + tp["beta"]
        # first day of the target month is deterministic: it uses the last OBSERVED return,
        # scaled by tau of the target month (the EGS within-month convention)
        g1 = (1.0 - rho) + tp["alpha"] * resid_last_sq / tau_target + tp["beta"] * g_last
        s = expected_g_sum(g1, rho, int(n_days_target))
        return float(tau_target * s + n_days_target * tp["mu"] ** 2)

print("Model estimator ready.")

Model estimator ready.


## Expanding-window backtest

Define the expanding-window fitting paths, then refit both models monthly using only information available at each forecast origin. GARCH-MIDAS uses that month’s GARCH estimate as one of its optimiser starting values.

In [6]:
def fit_at_origin(use_theta, macro_np, rets, origin, nested_params=None, seed=None):
    """The GARCH(1,1) fitting path, shared by the walk-forward and the leakage audit.

    GARCH has no parameter shared across ETFs, so its likelihood is a sum of five independent
    ETF terms and the joint fit returns each ETF's own maximum-likelihood estimates (checked in
    Step 6). Fits on daily excess returns of months K..origin (and macro months 0..origin).
    Starting points: N_STARTS fixed ones, N_RANDOM random ones (seeded by the origin month),
    and for GARCH-MIDAS this month's GARCH(1,1) solution with theta = 0 (nested_params).
    Since GARCH-MIDAS nests GARCH, that start guarantees the GARCH-MIDAS fit is at least as
    good as GARCH's. Each month's fit uses only that month's data window, so months do not
    depend on one another.
    """
    n_obs = origin + 1
    lo, hi = slice_bounds(K_MIDAS, n_obs)
    m = PooledGarchMidas(K_MIDAS, TICKERS, use_theta=use_theta, n_starts=N_STARTS,
                         seed=(100_000 + origin) if seed is None else seed, n_random=N_RANDOM)
    extras = []
    if use_theta and nested_params is not None:
        extras.append(m.params_to_z(nested_params, theta=0.0))
    m.fit(rets, MIDS, lo, hi, macro_np[:n_obs], n_obs, extra_z0=extras or None)
    return m


def _cache_path(tag):
    return os.path.join(CACHE_DIR, hashlib.md5(tag.encode()).hexdigest() + ".json")


def cache_get(tag):
    if not CACHE_ENABLED:
        return None
    p = _cache_path(tag)
    if not os.path.exists(p):
        return None
    try:
        with open(p) as f:
            obj = json.load(f)
        return obj if obj.get("tag") == tag else None      # guard against hash collisions
    except Exception:
        return None


def cache_put(tag, obj):
    if not CACHE_ENABLED:
        return
    obj = dict(obj, tag=tag)
    tmp = _cache_path(tag) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, _cache_path(tag))          # atomic: an interrupted write leaves no stub


def spec_for(label, use_theta):
    return (f"phase2|{label}|theta={use_theta}|K={K_MIDAS}|burn={BURN_IN}|refit={REFIT_EVERY}"
            f"|grid={GRID[0]}..{GRID[-1]}|{TICKERS}|starts={N_STARTS}+{N_RANDOM}rand+nested+polish-all"
            f"|data={DATA_FINGERPRINT}")


def fit_record(use_theta, macro_np, origin, nested=None):
    """Fit one month and return what is saved in the cache."""
    m = fit_at_origin(use_theta, macro_np, RETS, origin, nested_params=nested)
    return dict(params=m.params, fit_ok=bool(m.fit_ok),
                loglik=(float(m.loglik) if np.isfinite(m.loglik) else None),
                conv_gap=float(m.conv_gap))


def walk_forward(label, macro_np, use_theta, nested_by_origin=None, last_origin=None):
    """Expanding window, re-estimated every REFIT_EVERY months, one-month-ahead forecasts.

    At origin T the model is fitted on months K..T, then forecasts target month T+1.
    If a refit fails, the previous refit's parameters are carried forward (information
    dated <= T only) and the event is recorded -- never silent. Refits are cached, so an
    interrupted run resumes exactly where it stopped.
    Returns (forecast rows, refit log, {origin: params}).
    """
    spec = spec_for(label, use_theta)
    stop = LAST_ORIGIN if last_origin is None else last_origin
    rows, refits, params_by_origin = [], [], {}
    active, active_origin = None, None
    for origin in range(FIRST_ORIGIN, stop + 1):
        n_obs = origin + 1
        lo, hi = slice_bounds(K_MIDAS, n_obs)
        if (origin - FIRST_ORIGIN) % REFIT_EVERY == 0:
            tag = f"{spec}|origin={origin}"
            hit = cache_get(tag)
            if hit is None:
                nested = nested_by_origin.get(origin) if (use_theta and nested_by_origin) else None
                if use_theta and nested is None:
                    raise RuntimeError("GARCH-MIDAS needs this month's GARCH solution (run GARCH first)")
                hit = fit_record(use_theta, macro_np, origin, nested)
                cache_put(tag, hit)
            carried = not (hit["fit_ok"] and hit["params"] is not None)
            if not carried:
                active, active_origin = hit["params"], origin
            elif active is None:
                raise RuntimeError(f"{label}: first refit at {GRID[origin]} failed to converge")
            refits.append(dict(model=label, origin=str(GRID[origin]), est_months=n_obs - K_MIDAS,
                               fit_ok=hit["fit_ok"], carried_forward=carried, loglik=hit["loglik"],
                               conv_gap=hit["conv_gap"],
                               theta=active["theta"], omega2=active["omega2"]))
        params_by_origin[origin] = active
        model = PooledGarchMidas(K_MIDAS, TICKERS, use_theta=use_theta).set_params(active)
        for k, tk in enumerate(TICKERS):
            nd = estimate_sessions(origin + 1, OBS[k])
            fc = model.forecast_h1(tk, RETS, MIDS, lo[k], hi[k], macro_np[:n_obs], origin, nd)
            rows.append(dict(model=label, etf=tk, origin=GRID[origin], target=GRID[origin + 1],
                             rv2_hat=fc, n_days_target=nd, theta=active["theta"],
                             params_from=str(GRID[active_origin])))
    return pd.DataFrame(rows), pd.DataFrame(refits), params_by_origin


def midas_tag(tk, origin):
    return f"{spec_for(MIDAS_CACHE_KEY, True)}|etf={tk}|origin={origin}"


def fit_midas_etf(tk, origin, macro_np, rets, garch_params, seed=None):
    """THE GARCH-MIDAS fitting path for one ETF at one origin (walk-forward and leakage audit).

    Every parameter, theta included, is specific to this ETF. Starting points: N_STARTS fixed
    ones, N_RANDOM random ones (seeded by ETF and origin), and this ETF's GARCH(1,1) solution
    with theta = 0. Because GARCH-MIDAS nests GARCH, that start guarantees the ETF's GARCH-MIDAS
    fit is at least as good as its GARCH fit.
    """
    k = TICKERS.index(tk)
    n_obs = origin + 1
    lo, hi = slice_bounds(K_MIDAS, n_obs)
    m = PooledGarchMidas(K_MIDAS, [tk], use_theta=True, n_starts=N_STARTS,
                         seed=(100_000 + 1_000 * (k + 1) + origin) if seed is None else seed,
                         n_random=N_RANDOM)
    nested = dict(per_ticker={tk: garch_params["per_ticker"][tk]}, theta=0.0, omega2=garch_params["omega2"])
    m.fit(rets, MIDS, lo[k:k + 1].copy(), hi[k:k + 1].copy(), macro_np[:n_obs], n_obs,
          extra_z0=[m.params_to_z(nested, theta=0.0)])
    return m


def etf_garch_loglik(tk, origin, garch_params, rets=None):
    """One ETF's term of the GARCH log-likelihood at the given parameters."""
    rets = RETS if rets is None else rets
    k = TICKERS.index(tk)
    lo, hi = slice_bounds(K_MIDAS, origin + 1)
    g = PooledGarchMidas(K_MIDAS, [tk], use_theta=False)
    z = g.params_to_z(dict(per_ticker={tk: garch_params["per_ticker"][tk]}, theta=0.0,
                           omega2=garch_params["omega2"]))
    return -nll_pooled(z, rets, MIDS, lo[k:k + 1].copy(), hi[k:k + 1].copy(), np.zeros(origin + 1),
                       K_MIDAS, origin + 1, 1, 0)


def walk_forward_midas(macro_np, garch_par, last_origin=None):
    """GARCH-MIDAS for each ETF separately: expanding window, refit every month, h = 1.

    Returns (forecast rows, refit log with one row per ETF and origin, {(etf, origin): params}).
    A failed fit would carry that ETF's previous parameters forward and be recorded.
    """
    stop = LAST_ORIGIN if last_origin is None else last_origin
    rows, refits, params = [], [], {}
    for k, tk in enumerate(TICKERS):
        active, active_origin = None, None
        for origin in range(FIRST_ORIGIN, stop + 1):
            n_obs = origin + 1
            hit = cache_get(midas_tag(tk, origin))
            if hit is None:
                m = fit_midas_etf(tk, origin, macro_np, RETS, garch_par[origin])
                hit = dict(params=m.params, fit_ok=bool(m.fit_ok),
                           loglik=float(m.loglik) if np.isfinite(m.loglik) else None,
                           conv_gap=float(m.conv_gap))
                cache_put(midas_tag(tk, origin), hit)
            carried = not (hit["fit_ok"] and hit["params"] is not None)
            if not carried:
                active, active_origin = hit["params"], origin
            elif active is None:
                raise RuntimeError(f"{tk}: first GARCH-MIDAS fit at {GRID[origin]} failed")
            ll_g = etf_garch_loglik(tk, origin, garch_par[origin])
            refits.append(dict(model=MIDAS_LABEL, etf=tk, origin=str(GRID[origin]), est_months=n_obs - K_MIDAS,
                               fit_ok=hit["fit_ok"], carried_forward=carried, loglik=hit["loglik"],
                               garch_loglik=ll_g, lr_vs_garch=2 * (hit["loglik"] - ll_g),
                               conv_gap=hit["conv_gap"], theta=active["theta"], omega2=active["omega2"]))
            params[(tk, origin)] = active
            lo, hi = slice_bounds(K_MIDAS, n_obs)
            model = PooledGarchMidas(K_MIDAS, [tk], use_theta=True).set_params(active)
            nd = estimate_sessions(origin + 1, OBS[k])
            fc = model.forecast_h1(tk, RETS, MIDS, lo[k], hi[k], macro_np[:n_obs], origin, nd)
            rows.append(dict(model=MIDAS_LABEL, etf=tk, origin=GRID[origin], target=GRID[origin + 1],
                             rv2_hat=fc, n_days_target=nd, theta=active["theta"],
                             params_from=str(GRID[active_origin])))
    return pd.DataFrame(rows), pd.DataFrame(refits), params


print("Backtest fitting routines ready.")

Backtest fitting routines ready.


In [7]:
FC, REF, PAR = {}, {}, {}
FC["garch"], REF["garch"], PAR["garch"] = walk_forward("garch", np.zeros(N_MONTHS), use_theta=False)
FC[MIDAS_LABEL], REF[MIDAS_LABEL], PAR[MIDAS_LABEL] = walk_forward_midas(
    macro_matrix[DRIVER].to_numpy(float), PAR["garch"]
)

_gap = REF[MIDAS_LABEL].loglik - REF[MIDAS_LABEL].garch_loglik
assert _gap.min() >= -1e-6, "GARCH-MIDAS fitted worse than GARCH"
assert max(REF["garch"].conv_gap.max(), REF[MIDAS_LABEL].conv_gap.max()) < 1e-6,     "a fit did not reach its maximum"
for _lab, _f in FC.items():
    assert len(_f) == len(TARGETS) * len(TICKERS), f"{_lab}: wrong number of forecasts"
    assert np.isfinite(_f.rv2_hat).all() and (_f.rv2_hat > 0).all(), f"{_lab}: bad forecast"
    assert ((_f.target - _f.origin).apply(lambda d: d.n) == HORIZON).all(), f"{_lab}: horizon"
    for _tk in TICKERS:
        assert list(_f[_f.etf == _tk].target) == list(TARGETS), f"{_lab}/{_tk}: target set"
print(f"Backtest complete: {len(TARGETS)} forecasts per ETF; {sum(r.fit_ok.sum() for r in REF.values())}/{sum(len(r) for r in REF.values())} fits converged.")


Backtest complete: 214 forecasts per ETF; 1284/1284 fits converged.


## Export forecasts

Create the tidy forecast--outcome file used for separate evaluation and a companion theta file for diagnosing the GARCH-MIDAS macro component. `date` is the forecast-origin month-end; the realised variance fields belong to the following target month.

In [8]:
EN_COLS = ["model", "etf", "date", "actual_logRV", "actual_RV", "fc_log", "sigma2", "fc_level"]

parts = []
for _lab in MODELS:
    f = FC[_lab].copy()
    f["date"] = [o.to_timestamp(how="end").strftime("%Y-%m-%d") for o in f.origin]
    f["actual_RV"] = [float(rv2.loc[t, tk]) for t, tk in zip(f.target, f.etf)]
    f["actual_logRV"] = np.log(f["actual_RV"])
    f["fc_level"] = f["rv2_hat"]
    f["fc_log"] = np.log(f["fc_level"])
    f["sigma2"] = np.nan
    f["etf"] = pd.Categorical(f["etf"], TICKERS, ordered=True)
    parts.append(f.sort_values(["etf", "origin"]))

detail = pd.concat(parts, ignore_index=True)
detail["etf"] = detail["etf"].astype(str)
tidy = detail[EN_COLS].copy()
expected_rows = len(MODELS) * len(TICKERS) * len(TARGETS)
assert len(tidy) == expected_rows
assert not tidy.duplicated(["model", "etf", "date"]).any()
assert set(tidy.model) == set(MODELS) and set(tidy.etf) == set(TICKERS)
assert tidy[["actual_RV", "actual_logRV", "fc_level", "fc_log"]].notna().all().all()
assert (tidy[["actual_RV", "fc_level"]] > 0).all().all()
tidy.to_csv(OUT_TIDY, index=False)

theta_out = FC[MIDAS_LABEL][["model", "etf", "origin", "target", "theta", "params_from"]].copy()
theta_out["date"] = [o.to_timestamp(how="end").strftime("%Y-%m-%d") for o in theta_out.origin]
theta_out["target_date"] = [t.to_timestamp(how="end").strftime("%Y-%m-%d") for t in theta_out.target]
theta_out = theta_out[["model", "etf", "date", "target_date", "theta", "params_from"]].sort_values(["etf", "date"])
assert len(theta_out) == len(TICKERS) * len(TARGETS)
assert not theta_out.duplicated(["etf", "date"]).any()
theta_out.to_csv(OUT_THETA, index=False)

print(f"Saved {len(tidy)} forecast rows to {os.path.basename(OUT_TIDY)} and {len(theta_out)} theta rows to {os.path.basename(OUT_THETA)}.")
display(tidy.head())


Saved 2140 forecast rows to garch_forecasts.csv and 1070 theta rows to garchmidas_theta_by_origin.csv.


,model,etf,date,actual_logRV,actual_RV,fc_log,sigma2,fc_level
0,garch,XLI,2008-02-29,-4.808556,0.008160,-5.657044,NaN,0.003493
1,garch,XLI,2008-03-31,-5.344238,0.004776,-5.018257,NaN,0.006616
2,garch,XLI,2008-04-30,-6.324658,0.001792,-5.315111,NaN,0.004917
3,garch,XLI,2008-05-31,-5.424881,0.004406,-5.927957,NaN,0.002664
4,garch,XLI,2008-06-30,-4.750961,0.008643,-5.447835,NaN,0.004306
